# Supplementary Figure 11 | S-LDSC annotation coverage, testis sensitivity, and attenuation of QTL heritability with baseline annotations.

**a.** Percent of 0.01 centimorgan (cM) blocks overlapped by each annotation, stratified by biotype and QTL type (eQTL, circles; sQTL, squares), for annotations built from all tissues (open markers) versus all tissues excluding testis (filled, biotype-colored; paired points connected by a vertical line). Biotypes are protein-coding (p.c.), all lncRNA, existing lncRNA, and novel lncRNA. Dashed line, 1.7% of 0.01 cM blocks, the minimum for well-calibrated S-LDSC p-values in heritability enrichment analyses.55 Because novel lncRNA sQTLs fall below this threshold (testis-excluded: 0.49%), they are not included in further S-LDSC analysis.

**b.** Meta-analyzed difference in stratified LD score regression (S-LDSC) heritability enrichment for the same biotypes and QTL types as (a), calculated as enrichment(excluding testis) − enrichment(all tissues). Points, inverse-variance-weighted (IVW) meta-analytic mean Δ enrichment across 87 GWAS traits; error bars, 95% confidence intervals (CIs). Dashed line, Δ = 0. Asterisks denote two-sided IVW significance of Δ enrichment ≠ 0 (*, p < 0.05; **, p < 0.01; ***, p < 0.001; ****, p < 0.0001). Black x, annotation omitted because 0.01 cM-block coverage is below the 1.7% threshold in (a). Positive values indicate higher enrichment after excluding testis; all remaining S-LDSC analyses exclude testis QTLs.

**c.** Meta-analyzed percent of conditional effect (τ*) lost with inclusion of each S-LDSC baseline track one at a time, calculated as 100 × (τ_QTL only − τ_QTL + baseline track) / τ_QTL only, for novel lncRNA, existing lncRNA, and protein-coding eQTL and sQTL annotations. Higher values indicate that more of the focal QTL conditional heritability is lost with the added track. The first column shows τ lost after conditioning on the full baselineLD v2.2 model (all tracks); the second column shows conditioning on GTEx protein-coding eQTL and sQTLs (p.c. e/sQTL). Remaining columns show individual baselineLD v2.2 baseline annotation tracks with ≥50% τ* lost in at least one biotype-QTL class, ordered by τ* lost for novel lncRNA eQTLs. Asterisks denote IVW meta-analysis significance of τ* > 0 (*, p < 0.05; **, p < 0.01; ***, p < 0.001; ****, p < 0.0001). Most lncRNA eQTL conditional heritability is accounted for by histone and other regulatory tracks, consistent with correlation with regulatory context.


## Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from scipy import stats

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 7.5)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

paired = sns.color_palette("Paired", 12)
gene_type_palette = {
    "novel_lncRNA": paired[5],
    "existing_lncRNA": paired[4],
    "protein_coding": paired[1],
    "lncRNA_all": sns.color_palette("Reds", 7)[5],
}
qtl_marker = {"eqtl": "o", "sqtl": "s"}

gene_set_order = [
    "protein_coding",
    "lncRNA_all",
    "existing_lncRNA",
    "novel_lncRNA",
]
gene_set_labels = {
    "protein_coding": "p.c.",
    "lncRNA_all": "all\nlncRNA",
    "existing_lncRNA": "existing\nlncRNA",
    "novel_lncRNA": "novel\nlncRNA",
}

biotypes_heat = ("novel_lncRNA", "existing_lncRNA", "protein_coding")
qtl_types = ("eqtl", "sqtl")
biotype_label = {
    "protein_coding": "p.c.",
    "existing_lncRNA": "existing\nlncRNA",
    "novel_lncRNA": "novel\nlncRNA",
}
qtl_label = {"eqtl": "eQTL", "sqtl": "sQTL"}
exclude_tracks = {"GTEx_eQTL_MaxCPP"}
min_pct_tau_lost = 50
min_frac_pct = 1.7

baseline_row_label = "All LDSC baselineLD v2.2"
pc_esqtl_row_label = "GTEx p.c. e/sQTL"
spacer_top_label = " "
spacer_bottom_label = "  "


## Paths

In [ ]:
ldsc_summary_file = Path("path/to/ldsc_summary.tsv")
models_manifest_file = Path("ldsc/refs/ldsc_models.tsv")
trait_abbr_file = Path("path/to/table_s2_abbreviations.txt")
gwas_meta_file = Path("path/to/barbeira_full_gwas_metadata.txt")
coverage_file = Path("path/to/annot_snp_cm_coverage.tsv")

source_data_dir = Path("source_data")
tau_lost_long_file = source_data_dir / "supp_figure_11_ldsc_tau_lost_long.tsv"
tau_lost_all_pct_file = source_data_dir / "supp_figure_11_ldsc_tau_lost_all_tracks_wide_pct.tsv"
tau_lost_all_p_file = source_data_dir / "supp_figure_11_ldsc_tau_lost_all_tracks_wide_p.tsv"
tau_lost_all_sig_file = source_data_dir / "supp_figure_11_ldsc_tau_lost_all_tracks_wide_significance.tsv"
tau_lost_fig_pct_file = source_data_dir / "supp_figure_11_ldsc_tau_lost_figure_wide_pct.tsv"
tau_lost_fig_p_file = source_data_dir / "supp_figure_11_ldsc_tau_lost_figure_wide_p.tsv"
tau_lost_fig_sig_file = source_data_dir / "supp_figure_11_ldsc_tau_lost_figure_wide_significance.tsv"

fig_dir = Path("figures")


## Load data


### Table S2 trait abbreviations


In [ ]:
s2_abbr = set()
with open(trait_abbr_file) as fh:
    fh.readline()
    for line in fh:
        parts = line.strip().split()
        if parts:
            s2_abbr.add(parts[-2])
print(f"Table S2 abbreviations: {len(s2_abbr)} traits")


### GWAS metadata and trait cohort


In [ ]:
gwas_meta = pd.read_csv(gwas_meta_file, sep="\t")
trait_labels = {}
for tag, code in zip(gwas_meta["Tag"], gwas_meta["new_abbreviation"]):
    if pd.isna(code) or code not in s2_abbr:
        continue
    trait_labels[str(tag)] = str(code)
trait_cohort = np.array(sorted(trait_labels))
print(f"Trait cohort: {len(trait_cohort)} GWAS")


### LDSC summary


In [ ]:
ldsc = pd.read_csv(ldsc_summary_file, sep="\t", low_memory=False)
for col in ("Enrichment", "Enrichment_std_error", "tau_star", "tau_star_se", "h2"):
    if col in ldsc.columns:
        ldsc[col] = pd.to_numeric(ldsc[col], errors="coerce")
ldsc = ldsc[ldsc["trait_id"].isin(trait_cohort)].copy()

# Same table for model_id lookups (enrichment / baseOnly / attenuation).
baseline = baseonly = baseplus = ldsc

df = ldsc.dropna(
    subset=["Enrichment", "Enrichment_std_error", "tau_star", "tau_star_se", "h2"]
).copy()
df = df[df["Enrichment_std_error"] > 0].drop_duplicates(
    subset=["trait_id", "model_id"], keep="last"
)
print(
    f"LDSC summary: {len(df):,} enrichment rows, {df['model_id'].nunique()} models "
    f"from {ldsc_summary_file.name}"
)


### LDSC model manifest (attenuation tracks)


In [ ]:
models = pd.read_csv(models_manifest_file, sep="\t", dtype=str).fillna("")
# Attenuation models: QTL + one baseline-LD track (non-empty baseline_track).
manifest = models[models["baseline_track"] != ""].copy()
print(f"Model manifest: {len(models):,} total; {len(manifest):,} attenuation (basePlusOneTrack)")

### Annotation coverage table

In [ ]:
cov = pd.read_csv(coverage_file, sep="\t")
print(f"Loaded {coverage_file.name} ({len(cov)} rows)")

## Run analysis


### Meta-analysis helpers


In [ ]:
def sig_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.0001:
        return "****"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def ivw_meta(sub, vcol="Enrichment", scol="Enrichment_std_error", null=1.0, one_sided_gt=False):
    y = pd.to_numeric(sub[vcol], errors="coerce").to_numpy(float)
    se = pd.to_numeric(sub[scol], errors="coerce").to_numpy(float)
    ok = np.isfinite(y) & np.isfinite(se) & (se > 0)
    y, se = y[ok], se[ok]
    if len(y) == 0:
        return None
    w = 1.0 / se ** 2
    mu = float(np.dot(w, y) / w.sum())
    if len(y) > 1:
        sig2 = max(float(np.dot(w, (y - mu) ** 2) / (len(y) - 1)), 0.0)
        semu = sig2 ** 0.5 / w.sum() ** 0.5
    else:
        semu = float(se[0])
    z1 = (mu - null) / semu if semu > 0 else np.nan
    if one_sided_gt:
        p1 = float(stats.norm.sf(z1)) if np.isfinite(z1) else np.nan
    else:
        p1 = float(2 * stats.norm.sf(abs(z1))) if np.isfinite(z1) else np.nan
    return dict(
        n=len(y), mu=mu, se=semu, z=z1,
        ci_lo=mu - 1.96 * semu, ci_hi=mu + 1.96 * semu,
        p_vs1=p1 if null == 1.0 and not one_sided_gt else np.nan,
        p_gt0=p1 if null == 0.0 and one_sided_gt else np.nan,
        p=p1,
    )


def paired_diff_meta(mid_a, mid_b, null=0.0):
    cols = ["trait_id", "Enrichment", "Enrichment_std_error"]
    sub_a = df.loc[df["model_id"] == mid_a, cols].rename(
        columns={"Enrichment": "y_a", "Enrichment_std_error": "se_a"}
    )
    sub_b = df.loc[df["model_id"] == mid_b, cols].rename(
        columns={"Enrichment": "y_b", "Enrichment_std_error": "se_b"}
    )
    merged = sub_a.merge(sub_b, on="trait_id", how="inner")
    merged["Enrichment"] = merged["y_a"] - merged["y_b"]
    merged["Enrichment_std_error"] = np.sqrt(merged["se_a"] ** 2 + merged["se_b"] ** 2)
    return ivw_meta(merged, null=null, one_sided_gt=False)


def tissue_model_id(cat, q, excl_testis=False):
    return f"single_{cat}_{q}{'_noTestis' if excl_testis else ''}"


def baseonly_model_id(biotype, qtl):
    return f"single_{biotype}_{qtl}_noTestis_baseOnly"


def pc_esqtl_model_id(biotype, qtl):
    if biotype == "protein_coding":
        if qtl == "sqtl":
            return "single_protein_coding_esqtl_noTestis_baseOnly_focal_sqtl"
        return "single_protein_coding_esqtl_noTestis_baseOnly"
    return f"joint_{biotype}_vs_protein_coding_esqtl_noTestis_baseOnly_focal_{qtl}"


def baseline_model_id(biotype, qtl):
    return f"single_{biotype}_{qtl}_noTestis"


def col_label(biotype, qtl):
    return f"{biotype_label[biotype]}\n{qtl_label[qtl]}"


col_order = [col_label(b, q) for b in biotypes_heat for q in qtl_types]
novel_eqtl_col = col_label("novel_lncRNA", "eqtl")


### Annotation coverage (panel a)


In [ ]:
tissue_set_order = ["all_tissues", "excl_testis"]
idx = pd.MultiIndex.from_product(
    [gene_set_order, ["eqtl", "sqtl"], tissue_set_order],
    names=["biotype", "qtl", "tissues"],
)
plot_df = cov.set_index(["biotype", "qtl", "tissues"]).reindex(idx).reset_index()
missing = plot_df["pct_001cm"].isna().sum()
if missing:
    raise ValueError(
        f"Coverage table missing {missing} gene-set×QTL×tissues rows: {coverage_file}"
    )
b_skip = {
    (bt, q)
    for bt in gene_set_order
    for q in ["eqtl", "sqtl"]
    if plot_df.loc[(plot_df["biotype"] == bt) & (plot_df["qtl"] == q), "pct_001cm"].min(skipna=True)
    < min_frac_pct
}
print(f"Coverage: {len(cov)} rows; omitted panel b pairs: {len(b_skip)}")


### Conditional τ* % lost (panel c)


In [ ]:
baseonly_mu = {}
baseline_mu = {}
baseline_p_gt0 = {}
for b in biotypes_heat:
    for q in qtl_types:
        sub = baseonly[baseonly["model_id"] == baseonly_model_id(b, q)].dropna(
            subset=["tau_star", "tau_star_se", "h2"]
        )
        sub = sub[(sub["tau_star_se"] > 0) & (sub["h2"] > 0)]
        m = ivw_meta(sub, vcol="tau_star", scol="tau_star_se", null=0.0, one_sided_gt=True)
        baseonly_mu[(b, q)] = m["mu"] if m is not None else np.nan

        sub = baseline[baseline["model_id"] == baseline_model_id(b, q)].dropna(
            subset=["tau_star", "tau_star_se", "h2"]
        )
        sub = sub[(sub["tau_star_se"] > 0) & (sub["h2"] > 0)]
        m = ivw_meta(sub, vcol="tau_star", scol="tau_star_se", null=0.0, one_sided_gt=True)
        baseline_mu[(b, q)] = m["mu"] if m is not None else np.nan
        baseline_p_gt0[(b, q)] = m["p_gt0"] if m is not None else np.nan

rows = []
for _, mrow in manifest.iterrows():
    mid = mrow["model_id"]
    track = mrow["baseline_track"]
    primary = mrow["primary_annotation"]
    if primary.endswith("_eQTL_noTestis"):
        qtl = "eqtl"
        biotype = primary[: -len("_eQTL_noTestis")]
    elif primary.endswith("_sQTL_noTestis"):
        qtl = "sqtl"
        biotype = primary[: -len("_sQTL_noTestis")]
    else:
        continue
    sub = baseplus[baseplus["model_id"] == mid].dropna(subset=["tau_star", "tau_star_se", "h2"])
    sub = sub[(sub["tau_star_se"] > 0) & (sub["h2"] > 0)]
    m = ivw_meta(sub, vcol="tau_star", scol="tau_star_se", null=0.0, one_sided_gt=True)
    mu_plus = m["mu"] if m is not None else np.nan
    p_gt0 = m["p_gt0"] if m is not None else np.nan
    mu_base = baseonly_mu.get((biotype, qtl), np.nan)
    frac_lost = (mu_base - mu_plus) / mu_base if mu_base > 0 else np.nan
    rows.append(
        {
            "baseline_track": track,
            "biotype": biotype,
            "qtl": qtl,
            "column": col_label(biotype, qtl),
            "tau_p_gt0": p_gt0,
            "frac_tau_lost": frac_lost,
        }
    )

delta = pd.DataFrame(rows)
delta["pct_tau_lost"] = 100 * delta["frac_tau_lost"]
print(f"Delta table: {len(delta):,} rows, {delta['baseline_track'].nunique()} tracks")


### Heatmap matrices (panel c)


In [ ]:
heat = delta.pivot(index="baseline_track", columns="column", values="pct_tau_lost")
heat = heat.reindex(columns=col_order)
heat = heat.drop(index=[t for t in exclude_tracks if t in heat.index], errors="ignore")

track_order_all = heat[novel_eqtl_col].sort_values(ascending=False).index.tolist()
track_mask = (heat >= min_pct_tau_lost).any(axis=1)
track_order = heat.loc[track_mask].sort_values(novel_eqtl_col, ascending=False).index.tolist()
heat = heat.reindex(index=track_order)

heat_p = delta.pivot(index="baseline_track", columns="column", values="tau_p_gt0")
heat_p = heat_p.reindex(index=track_order, columns=col_order)
heat_p = heat_p.drop(index=[t for t in exclude_tracks if t in heat_p.index], errors="ignore")

baseline_row = pd.DataFrame(
    {
        col_label(b, q): 100 * (baseonly_mu[(b, q)] - baseline_mu[(b, q)]) / baseonly_mu[(b, q)]
        if baseonly_mu[(b, q)] > 0
        else np.nan
        for b in biotypes_heat
        for q in qtl_types
    },
    index=[baseline_row_label],
).reindex(columns=col_order)

baseline_p = pd.DataFrame(
    {col_label(b, q): baseline_p_gt0[(b, q)] for b in biotypes_heat for q in qtl_types},
    index=[baseline_row_label],
).reindex(columns=col_order)

pc_esqtl_frac = {}
pc_esqtl_p = {}
for b in biotypes_heat:
    for q in qtl_types:
        mid = pc_esqtl_model_id(b, q)
        sub = baseonly[baseonly["model_id"] == mid]
        col = col_label(b, q)
        if sub.empty:
            raise ValueError(f"Missing p.c. e/sQTL model in baseOnly summary: {mid}")
        sub = sub.dropna(subset=["tau_star", "tau_star_se", "h2"])
        sub = sub[(sub["tau_star_se"] > 0) & (sub["h2"] > 0)]
        m = ivw_meta(sub, vcol="tau_star", scol="tau_star_se", null=0.0, one_sided_gt=True)
        mu_pc = m["mu"] if m is not None else np.nan
        mu_base = baseonly_mu[(b, q)]
        pc_esqtl_frac[col] = 100 * (mu_base - mu_pc) / mu_base if mu_base > 0 else np.nan
        pc_esqtl_p[col] = m["p_gt0"] if m is not None else np.nan

pc_esqtl_row = pd.DataFrame(pc_esqtl_frac, index=[pc_esqtl_row_label]).reindex(columns=col_order)
pc_esqtl_p_row = pd.DataFrame(pc_esqtl_p, index=[pc_esqtl_row_label]).reindex(columns=col_order)

spacer_top = pd.DataFrame(np.nan, index=[spacer_top_label], columns=col_order)
spacer_bottom = pd.DataFrame(np.nan, index=[spacer_bottom_label], columns=col_order)
heat_plot = pd.concat([baseline_row, spacer_top, pc_esqtl_row, spacer_bottom, heat])

annot_plot = pd.concat(
    [
        baseline_p.map(sig_stars),
        pd.DataFrame("", index=[spacer_top_label], columns=col_order),
        pc_esqtl_p_row.map(sig_stars),
        pd.DataFrame("", index=[spacer_bottom_label], columns=col_order),
        heat_p.map(sig_stars),
    ]
)

print(f"Heatmap: {heat.shape[0]} tracks × {heat.shape[1]} columns (≥{min_pct_tau_lost}% in any biotype/QTL)")


### Supplement table (τ* lost source data)


In [ ]:
heat_all = delta.pivot(index="baseline_track", columns="column", values="pct_tau_lost")
heat_all = heat_all.reindex(index=track_order_all, columns=col_order)
heat_p_all = delta.pivot(index="baseline_track", columns="column", values="tau_p_gt0")
heat_p_all = heat_p_all.reindex(index=track_order_all, columns=col_order)

supp_long = delta.copy()
supp_long["significance"] = supp_long["tau_p_gt0"].map(sig_stars)
supp_long = supp_long[
    [
        "baseline_track",
        "biotype",
        "qtl",
        "pct_tau_lost",
        "frac_tau_lost",
        "tau_p_gt0",
        "significance",
    ]
].sort_values(["baseline_track", "biotype", "qtl"])

ref_rows = []
for b in biotypes_heat:
    for q in qtl_types:
        col = col_label(b, q)
        ref_rows.append(
            {
                "baseline_track": baseline_row_label,
                "biotype": b,
                "qtl": q,
                "pct_tau_lost": baseline_row.loc[baseline_row_label, col],
                "frac_tau_lost": baseline_row.loc[baseline_row_label, col] / 100,
                "tau_p_gt0": baseline_p.loc[baseline_row_label, col],
                "significance": sig_stars(baseline_p.loc[baseline_row_label, col]),
            }
        )
        ref_rows.append(
            {
                "baseline_track": pc_esqtl_row_label,
                "biotype": b,
                "qtl": q,
                "pct_tau_lost": pc_esqtl_row.loc[pc_esqtl_row_label, col],
                "frac_tau_lost": pc_esqtl_row.loc[pc_esqtl_row_label, col] / 100,
                "tau_p_gt0": pc_esqtl_p_row.loc[pc_esqtl_row_label, col],
                "significance": sig_stars(pc_esqtl_p_row.loc[pc_esqtl_row_label, col]),
            }
        )

supp_long = pd.concat([pd.DataFrame(ref_rows), supp_long], ignore_index=True)

supp_wide_pct_all = heat_all.round(1)
supp_wide_p_all = heat_p_all
supp_wide_sig_all = heat_p_all.map(sig_stars)
supp_wide_pct_fig = heat_plot.round(1)
supp_wide_p_fig = pd.concat(
    [
        baseline_p,
        pd.DataFrame(np.nan, index=[spacer_top_label], columns=col_order),
        pc_esqtl_p_row,
        pd.DataFrame(np.nan, index=[spacer_bottom_label], columns=col_order),
        heat_p,
    ]
)
supp_wide_sig_fig = annot_plot.copy()

source_data_dir.mkdir(parents=True, exist_ok=True)
supp_long.to_csv(tau_lost_long_file, sep="\t", index=False, float_format="%.4f")
supp_wide_pct_all.to_csv(tau_lost_all_pct_file, sep="\t", float_format="%.1f")
supp_wide_p_all.to_csv(tau_lost_all_p_file, sep="\t", float_format="%.4g")
supp_wide_sig_all.to_csv(tau_lost_all_sig_file, sep="\t")
supp_wide_pct_fig.to_csv(tau_lost_fig_pct_file, sep="\t", float_format="%.1f")
supp_wide_p_fig.to_csv(tau_lost_fig_p_file, sep="\t", float_format="%.4g")
supp_wide_sig_fig.to_csv(tau_lost_fig_sig_file, sep="\t")

print(f"Supplement table: {len(supp_long):,} long rows; {supp_wide_pct_all.shape[0]} all-tracks wide rows")


## Make figure


In [ ]:
fig = plt.figure(figsize=(7.2, 5.0), dpi=300)
gs = fig.add_gridspec(2, 2, width_ratios=[1.3, 2], height_ratios=[1.0, 1.55], hspace=0.55, wspace=0.32)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, :])

ax = ax_a
cov_qtl_dx = {"eqtl": -0.05, "sqtl": 0.05}
for biotype in gene_set_order:
    color = gene_type_palette[biotype]
    for qtl in ["eqtl", "sqtl"]:
        x = gene_set_order.index(biotype) + cov_qtl_dx[qtl]
        marker = qtl_marker[qtl]
        ys = {}
        for tissues in ["all_tissues", "excl_testis"]:
            row = plot_df[
                (plot_df["biotype"] == biotype)
                & (plot_df["qtl"] == qtl)
                & (plot_df["tissues"] == tissues)
            ]
            if row.empty:
                continue
            y = row.iloc[0]["pct_001cm"]
            if np.isfinite(y):
                ys[tissues] = y
        if len(ys) == 2:
            ax.plot(
                [x, x],
                [ys["all_tissues"], ys["excl_testis"]],
                color=color, lw=0.8, zorder=1,
            )
        for tissues in ["all_tissues", "excl_testis"]:
            if tissues not in ys:
                continue
            mfc = "white" if tissues == "all_tissues" else color
            ax.scatter(
                [x], [ys[tissues]], marker=marker, s=9,
                edgecolor="k", facecolor=mfc, linewidth=0.35, zorder=2,
            )

ax.axhline(min_frac_pct, color="#888888", ls="--", lw=0.8, zorder=0)
for _x in range(1, len(gene_set_order)):
    ax.axvline(_x - 0.5, color="0.75", ls="-", lw=0.5, zorder=0)
ax.set_xticks(range(len(gene_set_order)))
ax.set_xticklabels([gene_set_labels[bt] for bt in gene_set_order])
ax.set_ylabel("% of 0.01 cM blocks\noverlapped")
ymax_cov = max(plot_df["pct_001cm"].dropna().max() * 1.15, min_frac_pct * 2)
ax.set_ylim(0, ymax_cov)
ax.set_xlim(-0.55, len(gene_set_order) - 0.45)
ax.tick_params(axis="x", length=0)
ax.set_yticks([0, 5, 10, 15, 20])
ax.spines[["top", "right"]].set_visible(False)
ax.legend(
    handles=[
        Line2D([], [], marker="o", ls="", markerfacecolor="grey", markeredgecolor="k",
               markeredgewidth=0.35, ms=3, label="eQTL"),
        Line2D([], [], marker="s", ls="", markerfacecolor="grey", markeredgecolor="k",
               markeredgewidth=0.35, ms=3, label="sQTL"),
        Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor="k",
               markeredgewidth=0.35, ms=3, label="All tissues"),
        Line2D([], [], marker="o", ls="", markerfacecolor="grey", markeredgecolor="k",
               markeredgewidth=0.35, ms=3, label="Excl. Testis"),
    ],
    frameon=False, loc="center", ncol=2, bbox_to_anchor=(0.5, 1.12),
)

ax = ax_b
qtl_dx = {"eqtl": -0.18, "sqtl": 0.18}
b_extrema = []
for gi, cat in enumerate(gene_set_order):
    color = gene_type_palette[cat]
    for q in ["eqtl", "sqtl"]:
        x = gi + qtl_dx[q]
        if (cat, q) in b_skip:
            ax.plot(x, 0.0, marker="x", ms=3, mew=0.8, color="black", ls="", zorder=2)
            continue
        d = paired_diff_meta(
            tissue_model_id(cat, q, excl_testis=True),
            tissue_model_id(cat, q, excl_testis=False),
        )
        if d is None:
            continue
        ax.errorbar(
            [x], [d["mu"]],
            yerr=[[d["mu"] - d["ci_lo"]], [d["ci_hi"] - d["mu"]]],
            fmt=qtl_marker[q], ms=3, color=color, ecolor=color,
            markerfacecolor=color, markeredgecolor="k", markeredgewidth=0.35,
            capsize=0, lw=0.8, zorder=2,
        )
        b_extrema.extend([d["ci_lo"], d["ci_hi"]])
        star = sig_stars(d["p"])
        if star:
            star_y = d["ci_hi"] + 0.02
            ax.text(x, star_y, star, ha="center", va="bottom", color=color, fontsize=3.5)
            b_extrema.append(star_y + 0.05)

ax.axhline(0.0, color="#888888", ls="--", lw=0.8, zorder=0)
for _x in range(1, len(gene_set_order)):
    ax.axvline(_x - 0.5, color="0.75", ls="-", lw=0.5, zorder=0)
ax.set_xticks(range(len(gene_set_order)))
ax.set_xticklabels([gene_set_labels[bt] for bt in gene_set_order], rotation=0, ha="center")
ax.set_xlim(-0.55, len(gene_set_order) - 0.45)
ax.set_ylabel("Δ heritability enrichment\n(excl. Testis – all tissues)")
ax.tick_params(axis="x", length=0)
ax.spines[["top", "right"]].set_visible(False)
if b_extrema:
    lo, hi = min(b_extrema), max(b_extrema)
    pad = max((hi - lo) * 0.12, 0.05)
    ax.set_ylim(lo - pad, hi + pad)
ax.legend(
    handles=[
        Line2D([], [], marker="o", ls="", markerfacecolor="grey", markeredgecolor="k",
               markeredgewidth=0.35, ms=3, label="eQTL"),
        Line2D([], [], marker="s", ls="", markerfacecolor="grey", markeredgecolor="k",
               markeredgewidth=0.35, ms=3, label="sQTL"),
    ],
    frameon=False, loc="center", ncol=2, bbox_to_anchor=(0.5, 1.05),
)

heat_plot_rot = heat_plot.T
annot_plot_rot = annot_plot.T
spacer_labels = {spacer_top_label, spacer_bottom_label}
xtick_idx = [i for i, lab in enumerate(heat_plot.index) if lab not in spacer_labels]
xticklabels_rot = [heat_plot.index[i] for i in xtick_idx]

sns.heatmap(
    heat_plot_rot,
    ax=ax_c,
    cmap="Reds",
    vmin=0,
    vmax=100,
    linewidths=0,
    annot=annot_plot_rot,
    fmt="",
    annot_kws={"rotation": 90, "fontsize": 4},
    cbar_kws={"label": "% τ* lost (QTL − QTL|track)/QTL × 100", "shrink": 0.85},
    yticklabels=col_order,
    xticklabels=False,
)
ax_c.set_xticks([i + 0.5 for i in xtick_idx])
ax_c.set_xticklabels(xticklabels_rot, rotation=90, ha="center", va="top", fontsize=5)
ax_c.set_xlabel("")
ax_c.set_ylabel("")
ax_c.tick_params(axis="x", length=3, labelsize=4)
ax_c.tick_params(axis="y", labelsize=5)

_inv = fig.transFigure.inverted()
for ax, label, y_ax in [(ax_a, "a", 1.18), (ax_b, "b", 1.18), (ax_c, "c", 1.02)]:
    _x, _y = _inv.transform(ax.transAxes.transform((-0.08, y_ax)))
    fig.text(_x, _y, label, fontweight="bold", va="top", ha="right", fontsize=9)

plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_11_ldsc_sensitivity.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_11_ldsc_sensitivity.png", bbox_inches="tight")
print("Wrote", fig_dir / "supp_figure_11_ldsc_sensitivity.pdf")
print("Wrote", fig_dir / "supp_figure_11_ldsc_sensitivity.png")

## Print stats


In [ ]:
# Panel a: novel lncRNA sQTL coverage, testis excluded
novel_sqtl_cov = plot_df[
    (plot_df["biotype"] == "novel_lncRNA")
    & (plot_df["qtl"] == "sqtl")
    & (plot_df["tissues"] == "excl_testis")
]["pct_001cm"].iloc[0]
print(
    f"Novel lncRNA sQTL (testis-excluded) 0.01 cM block coverage: "
    f"{novel_sqtl_cov:.2f}% (threshold {min_frac_pct}%)"
)

# Panel c / Results: % tau* lost to active chromatin tracks for lncRNA eQTLs
active_tracks = ["H3K9ac_Trynka", "H3K4me1_Trynka", "TFBS_ENCODE", "DGF_ENCODE"]
lnc_eqtl_cols = [col_label("novel_lncRNA", "eqtl"), col_label("existing_lncRNA", "eqtl")]
active_pct = heat_all.loc[active_tracks, lnc_eqtl_cols]
print(
    f"Active chromatin tracks account for "
    f"{active_pct.min().min():.0f}–{active_pct.max().max():.0f}% of lncRNA eVariant tau* attenuation"
)
